Analyses two supplementary dimensions of the pipeline bias:

  1. THREAT SCORE CONTAMINATION
     Tests whether case_threat_level varies by pipeline within the same
     criminality category. A clean threat score should be independent of
     the apprehension method — it should measure individual threat, not
     the data source.

     Available for 261,476 records (36.6% of total). Missingness is noted
     as a limitation — non-random missing threat scores could affect
     the generalizability of this analysis.

  2. TIME TREND: CAP EXPANSION 2022–2025
     Documents the dramatic shift in pipeline composition across the
     analysis period and its relationship to overall deportation rates.
     Shows that the structural bias documented in 01/02 was amplified
     by a policy decision to expand database-driven enforcement.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
from scipy import stats
from config import DATA_INTERIM, RESULTS_P2

CRIM_ORDER = [
    "1 Convicted Criminal",
    "2 Pending Criminal Charges",
    "3 Other Immigration Violator",
]


def load() -> pd.DataFrame:
    path = DATA_INTERIM / "arrests_classified.csv"
    if not path.exists():
        raise FileNotFoundError("Run 01_classify_pipelines.py first.")
    return pd.read_csv(path, low_memory=False)


def threat_score_analysis(df: pd.DataFrame):

Within identical criminality categories, does threat level vary by pipeline?

In [ ]:
    print("="*60)
    print("THREAT SCORE CONTAMINATION ANALYSIS")
    print("="*60)

    has_threat = df[df["case_threat_level"].notna()].copy()
    total = len(df)
    with_threat = len(has_threat)
    print(f"\nRecords with threat level: {with_threat:,} / {total:,} "
          f"({with_threat/total:.1%})")
    print(f"Records missing threat level: {total-with_threat:,} ({(total-with_threat)/total:.1%})")

    # Missing pattern analysis
    missing_df = df.groupby("pipeline").apply(
        lambda x: pd.Series({
            "n_total":         len(x),
            "n_with_threat":   x["case_threat_level"].notna().sum(),
            "pct_with_threat": x["case_threat_level"].notna().mean(),
        })
    ).reset_index()
    print("\nMissing threat scores by pipeline:")
    print(missing_df.to_string(index=False))
    missing_df.to_csv(RESULTS_P2 / "threat_missing_analysis.csv", index=False)

    # Main analysis
    rows = []
    for crim in CRIM_ORDER:
        sub_c = has_threat[has_threat["apprehension_criminality"] == crim]
        print(f"\n── {crim} ──")
        for pipe in ["CAP", "Community", "287g"]:
            sub_p = sub_c[sub_c["pipeline"] == pipe]
            if len(sub_p) < 100:
                continue
            for tl in [1.0, 2.0, 3.0]:
                pct = (sub_p["case_threat_level"] == tl).mean()
                rows.append({
                    "criminality":   crim,
                    "pipeline":      pipe,
                    "threat_level":  int(tl),
                    "pct":           round(pct, 4),
                    "n":             len(sub_p),
                })
            tl1_pct = (sub_p["case_threat_level"] == 1).mean()
            print(f"  {pipe}: Level 1 = {tl1_pct:.1%}  (n={len(sub_p):,})")

        # Key comparison for Convicted Criminal
        if crim == "1 Convicted Criminal":
            cap_tl1 = has_threat[
                (has_threat["apprehension_criminality"] == crim) &
                (has_threat["pipeline"] == "CAP")
            ]["case_threat_level"]
            bg_tl1  = has_threat[
                (has_threat["apprehension_criminality"] == crim) &
                (has_threat["pipeline"] == "287g")
            ]["case_threat_level"]
            if len(cap_tl1) > 0 and len(bg_tl1) > 0:
                cap_rate = (cap_tl1 == 1).mean()
                bg_rate  = (bg_tl1  == 1).mean()
                ratio    = cap_rate / bg_rate if bg_rate > 0 else np.nan
                print(f"\n  → CAP assigns Level 1 at {ratio:.2f}× the rate of 287(g)")
                print(f"    CAP: {cap_rate:.1%}  |  287(g): {bg_rate:.1%}")

    threat_df = pd.DataFrame(rows)
    threat_df.to_csv(RESULTS_P2 / "threat_level_by_pipeline.csv", index=False)
    print(f"\nSaved → {RESULTS_P2}/threat_level_by_pipeline.csv")


def time_trend_analysis(df: pd.DataFrame):


Documents the CAP expansion 2022-2025 and its relationship to overall deportation rates.
    


In [ ]:
    print("\n" + "="*60)
    print("TIME TREND: CAP EXPANSION 2022-2025")
    print("="*60)

    valid = df[df["year"].between(2022, 2025)].copy()

    trend = (
        valid.groupby("year")
        .apply(lambda x: pd.Series({
            "n_arrests":       len(x),
            "pct_cap":         round((x["pipeline"] == "CAP").mean(), 4),
            "pct_community":   round((x["pipeline"] == "Community").mean(), 4),
            "pct_287g":        round((x["pipeline"] == "287g").mean(), 4),
            "pct_border":      round((x["pipeline"] == "Border").mean(), 4),
            "deportation_rate": round(x["deported"].mean(), 4),
        }))
        .reset_index()
    )

    print(trend.to_string(index=False))

    # Key statistics
    cap_2022 = trend.loc[trend["year"] == 2022, "pct_cap"].values[0]
    cap_2024 = trend.loc[trend["year"] == 2024, "pct_cap"].values[0]
    dep_2022 = trend.loc[trend["year"] == 2022, "deportation_rate"].values[0]
    dep_2024 = trend.loc[trend["year"] == 2024, "deportation_rate"].values[0]

    print(f"\nCAP share:        {cap_2022:.1%} (2022) → {cap_2024:.1%} (2024)  "
          f"[+{(cap_2024-cap_2022)*100:.1f}pp]")
    print(f"Deportation rate: {dep_2022:.1%} (2022) → {dep_2024:.1%} (2024)  "
          f"[+{(dep_2024-dep_2022)*100:.1f}pp]")

    trend.to_csv(RESULTS_P2 / "time_trend.csv", index=False)
    print(f"Saved → {RESULTS_P2}/time_trend.csv")


def main():
    df = load()
    threat_score_analysis(df)
    time_trend_analysis(df)
    print(f"\nAll supplementary analyses saved to {RESULTS_P2}")


if __name__ == "__main__":
    main()